In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ================================
# 1. POSITIONAL ENCODING
# ================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [ ]:
# ================================
# 2. MULTI-HEAD ATTENTION
# ================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)

    def forward(self, q, k, v):
        batch_size = q.size(0)

        q = self.split_heads(self.Wq(q), batch_size)
        k = self.split_heads(self.Wk(k), batch_size)
        v = self.split_heads(self.Wv(v), batch_size)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.depth)
        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)
        output = output.transpose(1, 2).contiguous()

        output = output.view(batch_size, -1, self.d_model)
        return self.fc(output)


In [ ]:
# ================================
# 3. FEED FORWARD NETWORK
# ================================
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# ================================
# 4. ENCODER LAYER
# ================================
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_output = self.attention(x, x, x)
        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)

        return x

In [ ]:
# ================================
# 5. TRANSFORMER MODEL
# ================================
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=100):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)

        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)

In [ ]:
# ================================
# 6. TEST THE MODEL
# ================================
if __name__ == "__main__":
    vocab_size = 1000
    d_model = 64
    num_heads = 8
    d_ff = 256
    num_layers = 2

    model = Transformer(vocab_size, d_model, num_heads, d_ff, num_layers)

    sample_input = torch.randint(0, vocab_size, (2, 10))  # (batch, seq_len)
    output = model(sample_input)

    print("Input shape:", sample_input.shape)
    print("Output shape:", output.shape)

Input shape: torch.Size([2, 10])
Output shape: torch.Size([2, 10, 1000])
